# Member C: canonical coupon, churn and messaging data

One tenant, three disjoint source populations. Real dates and outcomes are preserved. Raw ML training remains separate. Run cells in order.


In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data_prep' / 'transform' / 'common.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the project folder or data_prep/transform.')
TRANSFORM_DIR = PROJECT_ROOT / 'data_prep' / 'transform'
sys.path.insert(0, str(TRANSFORM_DIR))

OUTPUT_DIR = PROJECT_ROOT / 'data_prep' / 'processed'
COUPON_OUT_DIR = OUTPUT_DIR / 'predicting-coupon-redemption'
CHURN_OUT_DIR = OUTPUT_DIR / 'e-commerce-customer-churn'
MESSAGING_OUT_DIR = OUTPUT_DIR / 'direct-messaging'
os.makedirs(OUTPUT_DIR, exist_ok=True)
from member_c import transform_coupon, transform_churn, transform_messaging
from contract import publish, stage_directory


In [2]:
def raw_dataset_path(handle, required_files):
    """Use local raw files when present; otherwise download from Kaggle."""
    dataset_dir = PROJECT_ROOT / 'data_prep' / 'raw' / handle.split('/')[-1]
    if all(any((dataset_dir / name).is_file() for name in alternatives)
           for alternatives in required_files):
        print(f'Using local dataset: {dataset_dir}')
        return dataset_dir
    import kagglehub
    print(f'Downloading dataset to: {dataset_dir}')
    return Path(kagglehub.dataset_download(handle, output_dir=str(dataset_dir)))

coupon_path = str(raw_dataset_path('vasudeva009/predicting-coupon-redemption', [('train.csv',), ('campaign_data.csv',), ('coupon_item_mapping.csv',), ('customer_demographics.csv',), ('customer_transaction_data.csv',), ('item_data.csv',)]))
churn_path = str(raw_dataset_path('samuelsemaya/e-commerce-customer-churn', [('data_ecommerce_customer_churn.csv',)]))
messaging_path = str(raw_dataset_path('mkechinov/direct-messaging', [('campaigns.csv',), ('messages-demo.csv',), ('client_first_purchase_date.csv',), ('holidays.csv',)]))

print('Coupon:', coupon_path)
print('Churn:', churn_path)
print('Messaging:', messaging_path)

Using local dataset: d:\CDP -FYP\data_prep\raw\predicting-coupon-redemption
Using local dataset: d:\CDP -FYP\data_prep\raw\e-commerce-customer-churn
Using local dataset: d:\CDP -FYP\data_prep\raw\direct-messaging
Coupon: d:\CDP -FYP\data_prep\raw\predicting-coupon-redemption
Churn: d:\CDP -FYP\data_prep\raw\e-commerce-customer-churn
Messaging: d:\CDP -FYP\data_prep\raw\direct-messaging


In [3]:
coupon_frames, coupon_raw, coupon_sidecars = transform_coupon(coupon_path)
print({t: len(f) for t, f in coupon_frames.items()})


{'stores': 1, 'customers': 1582, 'products': 74066, 'orders': 1324566, 'order_items': 1324566, 'coupon_campaigns': 28, 'coupon_redemptions': 78369}


In [4]:
coupon_manifest = publish(
    COUPON_OUT_DIR, coupon_frames, [Path(coupon_path) / n for n in coupon_raw],
    {n: len(f) for n, f in coupon_raw.items()}, sidecars=coupon_sidecars,
    expected_counts={'order_items': len(coupon_raw['customer_transaction_data.csv']),
                     'orders': len(coupon_raw['customer_transaction_data.csv']),
                     'coupon_redemptions': len(coupon_raw['train.csv'])},
    notes=['Orders are synthetic source-line containers, not observed baskets: basket counts/AOV unsupported.',
           'selling_price is preserved verbatim; gross/net interpretation and source currency remain unverified.',
           'coupon_discount maps to coupon_disc with original sign. No second discount subtraction.'],
)


In [5]:
churn_frames, churn_raw = transform_churn(churn_path)
churn_manifest = publish(
    CHURN_OUT_DIR, churn_frames, [Path(churn_path) / 'data_ecommerce_customer_churn.csv'],
    {'data_ecommerce_customer_churn.csv': len(churn_raw)},
    expected_counts={'customers': len(churn_raw)},
    notes=['Native snapshot features retained. Tenure unit and observation date unknown; no invented purchase dates.'],
)


In [6]:
messaging_stage = stage_directory(MESSAGING_OUT_DIR)
messaging_frames, messaging_counts, messages_csv = transform_messaging(messaging_path, messaging_stage)
messaging_manifest = publish(
    MESSAGING_OUT_DIR, messaging_frames, [Path(messaging_path) / n for n in messaging_counts],
    messaging_counts, stage=messaging_stage, streams={'message_sends': messages_csv},
    expected_counts={'message_sends': messaging_counts['messages-demo.csv']},
    notes=['Customer roster includes message-only customers.',
           'Purchase outcomes do not identify unrelated retail/coupon orders; revenue attribution unavailable.'],
)
print('Member C publications complete.')


Member C publications complete.
